Pre-LLM Pipeline for Technical Proposal Generation

This module handles:
- Retrieval from Qdrant (RFP / TP)
- Context building and cleaning
- Technical offer structure definition
- Mapping RFP content into a fixed technical proposal template

Output of this module is a structured draft ready for LLM refinement.



In [1]:
TECH_OFFER_STRUCTURE = [
    {"key": "intro",              "title": "1) مقدمة وفهم عام للمشروع",          "required": True},
    {"key": "scope",              "title": "2) فهم نطاق العمل والمتطلبات",        "required": True},
    {"key": "methodology",        "title": "3) منهجية التنفيذ وآلية العمل",       "required": True},
    {"key": "timeline",           "title": "4) خطة العمل والجدول الزمني",          "required": True},
    {"key": "deliverables",       "title": "5) المخرجات والتسليمات",              "required": True},

    {"key": "team",               "title": "6) فريق العمل والحوكمة",              "required": False},
    {"key": "quality",            "title": "7) ضمان الجودة ومعايير القبول",        "required": False},
    {"key": "risk",               "title": "8) إدارة المخاطر",                    "required": False},
    {"key": "assumptions",        "title": "9) الافتراضات والقيود",               "required": False},
    
    {"key": "compliance",         "title": "10) الالتزام والمتطلبات النظامية",     "required": True},
]


Mapping

In [2]:
TECH_OFFER_INTENTS = {
    "intro": {
        "rfp_intent": "استخرج وصف المشروع وأهدافه وخلفيته وسبب تنفيذ المشروع. ابحث عن أي فقرة تعطي صورة عامة عن المشروع.",
        "tp_intent":  "اجلب أسلوب كتابة مقدمة عرض فني رسمي: تعريف بالمشروع، الهدف، والنطاق العام."
    },
    "scope": {
        "rfp_intent": "استخرج نطاق العمل: المهام، الأنشطة، المتطلبات الفنية، وما المطلوب تنفيذه. ركّز على ما يجب تسليمه.",
        "tp_intent":  "اجلب أسلوب صياغة قسم فهم نطاق العمل والمتطلبات وكيف يتم عرضها بشكل منظم."
    },
    "methodology": {
        "rfp_intent": "استخرج أي توجيهات أو متطلبات تخص طريقة التنفيذ أو إدارة المشروع أو مراحل العمل.",
        "tp_intent":  "اجلب أسلوب كتابة المنهجية: مراحل، آلية عمل، إدارة مشروع، تواصل، تقارير."
    },
    "timeline": {
        "rfp_intent": "استخرج المدة الزمنية، الجدول الزمني، المراحل، التواريخ أو الأرباع/الأشهر، وأي متطلبات تسليم زمنية.",
        "tp_intent":  "اجلب أسلوب كتابة خطة عمل وجدول زمني في عرض فني (مراحل + مدة + تسليمات)."
    },
    "deliverables": {
        "rfp_intent": "استخرج قائمة المخرجات: وثائق، تقارير، أنظمة، تدريب، تسليمات نهائية ومعايير الاستلام.",
        "tp_intent":  "اجلب أسلوب كتابة قسم المخرجات والتسليمات وكيف يتم سردها بشكل احترافي."
    },
    "compliance": {
        "rfp_intent": "استخرج أي التزامات نظامية أو لائحية أو شروط امتثال أو معايير يجب الالتزام بها.",
        "tp_intent":  "اجلب أسلوب صياغة فقرة الالتزام والامتثال بشكل رسمي وواضح."
    }
}


# Technical Offer Draft Generator

This module generates a structured technical offer draft using:
- **RFP** as the main source of requirements
- **TP** as an optional writing-style reference (no copying)

Key features:
- General, reusable technical offer structure (not RFP-specific)
- Intent-based semantic retrieval from Qdrant
- Cleaned, deduplicated, and chunked evidence (no LLM)
- Clear separation between content (RFP) and writing style (TP)

The output follows common government proposal methodology and is safe for reuse across projects.


In [3]:
import re
from typing import Dict, List, Any, Optional
from qdrant_client.http import models as qm


def generate_tech_offer_draft_with_intents(
    pair_id: int,
    project_title: str,
    structure: List[Dict[str, Any]],
    client,
    collection_name: str,
    embed_fn,
    intents: Optional[Dict[str, Dict[str, str]]] = None,
    include_tp_support: bool = True,
    top_k: int = 8,
    min_score: float = 0.35,
    max_chars_rfp: int = 2500,
    max_chars_tp: int = 1200,
    min_chunk_chars: int = 80,
    dedup: bool = True,
    include_evidence_headers: bool = False,
    scope_text: str = "",
    timeline_text: str = "",
) -> str:
    """
    Generate a technical-offer draft (Markdown) using intent-based retrieval.

    - Uses structure (required=True only)
    - Retrieves from RFP as primary evidence
    - Optionally retrieves from TP as style support
    - Uses intents (descriptions) rather than exact keyword matching
    """

    # -----------------------------
    # 1) Default intents (if not provided)
    # -----------------------------
    DEFAULT_INTENTS = {
        "intro": {
            "rfp": "استخرج وصف المشروع وأهدافه وخلفيته وسبب تنفيذ المشروع. ركّز على أي فقرة تعطي صورة عامة عن المشروع.",
            "tp":  "اجلب أسلوب كتابة مقدمة عرض فني رسمي: تعريف بالمشروع، الهدف، والنطاق العام."
        },
        "scope": {
            "rfp": "استخرج نطاق العمل: المهام، الأنشطة، المتطلبات الفنية، وما المطلوب تنفيذه. ركّز على المطلوب تنفيذه وما يجب تسليمه.",
            "tp":  "اجلب أسلوب صياغة قسم فهم نطاق العمل والمتطلبات وكيف يتم عرضها بشكل منظم."
        },
        "methodology": {
            "rfp": "استخرج أي توجيهات أو متطلبات تخص طريقة التنفيذ أو إدارة المشروع أو مراحل العمل أو آلية التواصل والتقارير.",
            "tp":  "اجلب أسلوب كتابة المنهجية: مراحل، آلية عمل، إدارة مشروع، تواصل، تقارير."
        },
        "timeline": {
            "rfp": "استخرج المدة الزمنية، الجدول الزمني، المراحل، التواريخ أو الأرباع/الأشهر، وأي متطلبات تسليم زمنية.",
            "tp":  "اجلب أسلوب كتابة خطة عمل وجدول زمني في عرض فني (مراحل + مدة + تسليمات)."
        },
        "deliverables": {
            "rfp": "استخرج قائمة المخرجات: وثائق، تقارير، أنظمة، تدريب، تسليمات نهائية ومعايير الاستلام.",
            "tp":  "اجلب أسلوب كتابة قسم المخرجات والتسليمات وكيف يتم سردها بشكل احترافي."
        },
        "compliance": {
            "rfp": "استخرج أي التزامات نظامية أو لائحية أو شروط امتثال أو معايير يجب الالتزام بها ضمن تنفيذ المشروع.",
            "tp":  "اجلب أسلوب صياغة فقرة الالتزام والامتثال بشكل رسمي وواضح."
        }
    }

    intents = intents or DEFAULT_INTENTS

    # -----------------------------
    # 2) Helpers
    # -----------------------------
    def _clean(text: str) -> str:
        text = (text or "").strip()
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    def _fingerprint(text: str, n: int = 220) -> str:
        return _clean(text)[:n]

    def _signals_for_key(key: str) -> List[str]:
        # Optional "signal" keywords to enrich retrieval without requiring exact matches
        if key == "timeline":
            return ["مدة", "مرحلة", "شهر", "ربع", "تاريخ", "schedule", "timeline", "خطة"]
        if key == "deliverables":
            return ["مخرجات", "تسليمات", "تقارير", "وثائق", "تدريب", "استلام", "deliverables"]
        if key == "scope":
            return ["نطاق", "مهام", "متطلبات", "تنفيذ", "خدمات", "أعمال", "SOW"]
        if key == "methodology":
            return ["منهجية", "آلية", "إدارة", "خطة", "تنفيذ", "حوكمة", "تقارير"]
        if key == "intro":
            return ["هدف", "نبذة", "خلفية", "مقدمة", "وصف المشروع"]
        if key == "compliance":
            return ["التزام", "أنظمة", "لوائح", "متطلبات", "امتثال", "حوكمة"]
        return []

    def _intent_to_query(key: str, doc_type: str) -> str:
        # doc_type: "rfp" or "tp"
        base = intents.get(key, {}).get(doc_type, "")
        if not base:
            base = f"استخرج محتوى مناسب لقسم '{key}' من وثيقة '{doc_type}'."
        signals = " | ".join(_signals_for_key(key))
        return f"{base}\nSignals: {signals}".strip()

    def _retrieve(doc_type: str, query: str, limit: int) -> List[Dict[str, Any]]:
        qv = embed_fn([query])[0].tolist()

        flt = qm.Filter(
            must=[
                qm.FieldCondition(key="pair_id", match=qm.MatchValue(value=int(pair_id))),
                qm.FieldCondition(key="doc_type", match=qm.MatchValue(value=str(doc_type))),
            ]
        )

        res = client.query_points(
            collection_name=collection_name,
            query=qv,
            query_filter=flt,
            limit=limit,
            with_payload=True,
        )

        out = []
        for p in res.points:
            score = float(p.score or 0.0)
            if score < float(min_score):
                continue
            payload = p.payload or {}
            out.append({
                "score": score,
                "text": payload.get("text", ""),
                "chunk_id": payload.get("chunk_id"),
                "file": payload.get("file"),
            })
        return out

    def _build_context(hits: List[Dict[str, Any]], max_chars: int) -> str:
        total = 0
        parts = []
        seen = set()

        for i, h in enumerate(hits, 1):
            txt = (h.get("text") or "").strip()
            txt = re.sub(r"\s+", " ", txt).strip()
            if len(txt) < min_chunk_chars:
                continue

            if dedup:
                fp = _fingerprint(txt)
                if not fp or fp in seen:
                    continue
                seen.add(fp)

            header = ""
            if include_evidence_headers:
                header = f"[{i}] score={h.get('score',0.0):.3f} | file={h.get('file')} | chunk={h.get('chunk_id')}\n"

            block = header + txt + "\n\n"
            if total + len(block) > max_chars:
                break

            parts.append(block)
            total += len(block)

        return "".join(parts).strip()

    def _user_injection(key: str) -> str:
        if key == "scope" and scope_text.strip():
            return f"**مدخل المستخدم (نطاق العمل):**\n{scope_text.strip()}\n\n"
        if key == "timeline" and timeline_text.strip():
            return f"**مدخل المستخدم (الجدول الزمني):**\n{timeline_text.strip()}\n\n"
        return ""

    # -----------------------------
    # 3) Build markdown draft
    # -----------------------------
    md = []
    md.append("# العرض الفني\n")
    md.append(f"**اسم المشروع:** {project_title}\n")
    md.append(f"**رقم المشروع (pair_id):** {pair_id}\n")

    for sec in structure:
        # Skip optional sections
        if not sec.get("required", False):
            continue

        key = sec["key"]
        title = sec["title"]

        # ---- Primary RFP
        q_rfp = _intent_to_query(key, "rfp")
        rfp_hits = _retrieve("rfp", q_rfp, limit=top_k)
        rfp_ctx = _build_context(rfp_hits, max_chars=max_chars_rfp)

        # ---- TP support (optional)
        tp_ctx = ""
        if include_tp_support:
            q_tp = _intent_to_query(key, "tp")
            tp_hits = _retrieve("tp", q_tp, limit=max(3, top_k // 2))
            tp_ctx = _build_context(tp_hits, max_chars=max_chars_tp)

        # Fallback
        if not rfp_ctx:
            rfp_ctx = "(لم يتم العثور على مقاطع كافية من الكراسة لهذا القسم. جرّبي رفع top_k أو خفض min_score أو تأكدي من pair_id.)"

        md.append(f"\n## {title}\n")
        md.append(_user_injection(key) + rfp_ctx)

        if tp_ctx:
            md.append("\n\n**مرجع أسلوبي (TP):**\n")
            md.append(tp_ctx)

    return "\n".join(md).strip()


C:\Users\yara2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from qdrant_client import QdrantClient

QDRANT_URL = "http://localhost:6333"  # أو اللي عندك
COLLECTION = "inspira_chunks_v1"      # نفس اسم الكولكشن

client = QdrantClient(url=QDRANT_URL)


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def embed(texts):
    return model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )


C:\Users\yara2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [6]:
import re
from typing import List, Dict, Any, Optional

def clean_ocr(text: str) -> str:
    if not text:
        return ""
    # OCR fixes (خفيفة)
    text = text.replace("اال", "ال").replace("ميالديه", "ميلادية")
    # normalize whitespace
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_paragraphs(text: str, min_len: int = 70, max_len: int = 450) -> List[str]:
    # split by blank lines first
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
    out: List[str] = []

    for b in blocks:
        if len(b) <= max_len:
            out.append(b)
        else:
            # split long blocks by sentence boundaries
            sents = re.split(r"(?<=[\.\!\?؟])\s+", b)
            cur = ""
            for s in sents:
                s = s.strip()
                if not s:
                    continue
                if not cur:
                    cur = s
                elif len(cur) + 1 + len(s) <= max_len:
                    cur += " " + s
                else:
                    out.append(cur.strip())
                    cur = s
            if cur:
                out.append(cur.strip())

    # filter short pieces
    out = [p.strip() for p in out if len(p.strip()) >= min_len]
    return out


In [7]:
import re
from typing import Dict, List, Any, Optional
from qdrant_client.http import models as qm


def generate_business_style_tech_offer(
    pair_id: int,
    project_title: str,
    structure: List[Dict[str, Any]],
    client,
    collection_name: str,
    embed_fn,
    intents: Optional[Dict[str, Dict[str, str]]] = None,
    include_tp_support: bool = True,
    top_k: int = 8,
    min_score: float = 0.35,
    max_chars_rfp: int = 2500,
    max_chars_tp: int = 900,
    min_chunk_chars: int = 80,
    dedup: bool = True,
    include_evidence: bool = True,
    scope_text: str = "",
    timeline_text: str = "",
) -> str:

    # ---------- Intents (fallback) ----------
    DEFAULT_INTENTS = {
        "intro": {"rfp": "استخرج وصف المشروع وأهدافه وخلفيته وسبب التنفيذ.", "tp": "اجلب أسلوب كتابة مقدمة عرض فني رسمي."},
        "scope": {"rfp": "استخرج نطاق العمل والمتطلبات الفنية والمهام والمخرجات.", "tp": "أسلوب كتابة فهم نطاق العمل والمتطلبات."},
        "methodology": {"rfp": "استخرج أي متطلبات لطريقة التنفيذ أو إدارة المشروع أو مراحل العمل.", "tp": "أسلوب كتابة المنهجية وإدارة المشروع."},
        "timeline": {"rfp": "استخرج المدة الزمنية والجدول الزمني والمراحل والتسليمات.", "tp": "أسلوب كتابة خطة العمل والجدول الزمني."},
        "deliverables": {"rfp": "استخرج المخرجات النهائية والتسليمات والتقارير ومعايير الاستلام.", "tp": "أسلوب كتابة قسم المخرجات والتسليمات."},
        "compliance": {"rfp": "استخرج أي التزامات نظامية/لوائح/شروط امتثال.", "tp": "أسلوب صياغة الالتزام والامتثال."},
    }
    intents = intents or DEFAULT_INTENTS

    # ---------- Helpers ----------
    def _clean(text: str) -> str:
        text = (text or "").strip()
        text = re.sub(r"\r\n", "\n", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]{2,}", " ", text)
        return text.strip()

    def _clean_ocr(text: str) -> str:
        """Light OCR/format cleanup (no rewriting)."""
        if not text:
            return ""
        text = text.replace("اال", "ال")
        text = text.replace("ميالديه", "ميلادية")
        text = text.replace("•", "- ").replace("–", "- ").replace("—", "- ")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    def _split_paragraphs(text: str, min_len: int = 70, max_len: int = 450) -> List[str]:
        """
        Smart-ish split:
        - split by blank lines
        - if block is long, split by sentence boundaries
        """
        blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
        out: List[str] = []

        for b in blocks:
            if len(b) <= max_len:
                out.append(b)
                continue

            # sentence split for long blocks
            sents = re.split(r"(?<=[\.\!\?؟])\s+", b)
            cur = ""
            for s in sents:
                s = s.strip()
                if not s:
                    continue
                if not cur:
                    cur = s
                elif len(cur) + 1 + len(s) <= max_len:
                    cur += " " + s
                else:
                    out.append(cur.strip())
                    cur = s
            if cur:
                out.append(cur.strip())

        # filter short
        out = [p.strip() for p in out if len(p.strip()) >= min_len]
        return out

    def _fingerprint(text: str, n: int = 220) -> str:
        return _clean(text)[:n]

    def _signals(key: str) -> List[str]:
        if key == "timeline":
            return ["مدة", "مرحلة", "شهر", "ربع", "تاريخ", "خطة", "timeline", "schedule"]
        if key == "deliverables":
            return ["مخرجات", "تسليمات", "تقارير", "وثائق", "تدريب", "استلام", "deliverables"]
        if key == "scope":
            return ["نطاق", "مهام", "متطلبات", "تنفيذ", "أعمال", "خدمات", "SOW"]
        if key == "methodology":
            return ["منهجية", "آلية", "إدارة", "حوكمة", "تقارير", "تنفيذ"]
        if key == "intro":
            return ["هدف", "نبذة", "خلفية", "مقدمة", "وصف"]
        if key == "compliance":
            return ["التزام", "أنظمة", "لوائح", "امتثال", "حوكمة"]
        return []

    def _intent_query(key: str, doc_type: str) -> str:
        base = intents.get(key, {}).get(doc_type, f"استخرج محتوى مناسب لقسم {key}")
        sig = " | ".join(_signals(key))
        return f"{base}\nSignals: {sig}".strip()

    def _retrieve(doc_type: str, query: str, limit: int) -> List[Dict[str, Any]]:
        qv = embed_fn([query])[0].tolist()
        flt = qm.Filter(
            must=[
                qm.FieldCondition(key="pair_id", match=qm.MatchValue(value=int(pair_id))),
                qm.FieldCondition(key="doc_type", match=qm.MatchValue(value=str(doc_type))),
            ]
        )
        res = client.query_points(
            collection_name=collection_name,
            query=qv,
            query_filter=flt,
            limit=limit,
            with_payload=True,
        )
        out = []
        for p in res.points:
            score = float(p.score or 0.0)
            if score < float(min_score):
                continue
            payload = p.payload or {}
            out.append({
                "score": score,
                "text": payload.get("text", ""),
                "chunk_id": payload.get("chunk_id"),
                "file": payload.get("file"),
            })
        return out

    # ✅ HERE: evidence context builder with cleaning + splitting + dedup at paragraph-level
    def _build_context(hits: List[Dict[str, Any]], max_chars: int) -> str:
        parts, seen, total = [], set(), 0

        for h in hits:
            raw = h.get("text", "")
            raw = _clean(raw)
            raw = _clean_ocr(raw)

            # split into short paragraphs
            paras = _split_paragraphs(raw, min_len=min_chunk_chars, max_len=450)

            for p in paras:
                if dedup:
                    fp = _fingerprint(p)
                    if not fp or fp in seen:
                        continue
                    seen.add(fp)

                block = f"- {p}\n"
                if total + len(block) > max_chars:
                    return "".join(parts).strip()

                parts.append(block)
                total += len(block)

        return "".join(parts).strip()

    # ---------- Business-style section writers ----------
    def _write_section(key: str, rfp_ctx: str, tp_ctx: str) -> str:
        user_scope = _clean(scope_text) if key == "scope" else ""
        user_time  = _clean(timeline_text) if key == "timeline" else ""

        if key == "intro":
            return (
                "نقدّم عرضنا الفني لتنفيذ المشروع وفق أفضل الممارسات، مع التركيز على تحقيق أهداف الجهة ورفع كفاءة التنفيذ وضمان جودة التسليم.\n"
                "يعتمد نهجنا على فهم واضح لمتطلبات الكراسة وتحويلها إلى خطة تنفيذ قابلة للقياس، مع حوكمة واضحة وإدارة مخاطر فعّالة.\n"
            )

        if key == "scope":
            base = (
                "### فهم نطاق العمل\n"
                "نقوم بتفكيك نطاق العمل إلى حزم عمل واضحة (Work Packages) وتحديد المتطلبات الفنية والتشغيلية لكل حزمة، "
                "مع توثيق المخرجات المتوقعة وآلية الاستلام.\n"
                "### ما سنقوم به\n"
                "- تحليل المتطلبات وتحديد حدود النطاق (In/Out of Scope)\n"
                "- بناء خطة تنفيذ تفصيلية للمخرجات والمهام\n"
                "- تحديد نقاط الاعتماد والتسليم\n"
            )
            if user_scope:
                base += f"\n**مدخل المستخدم (نطاق العمل):**\n{user_scope}\n"
            return base

        if key == "methodology":
            return (
                "### منهجية التنفيذ المقترحة\n"
                "نعتمد منهجية تنفيذ مرحلية لضمان التحكم بالجودة والوقت، وتشمل:\n"
                "1) بدء المشروع وتحليل الوضع الحالي والمتطلبات\n"
                "2) التخطيط التفصيلي (WBS) وتحديد الموارد والمخاطر\n"
                "3) التنفيذ على مراحل مع مراجعات واعتمادات دورية\n"
                "4) التحقق وضمان الجودة والتوثيق\n"
                "5) التسليم ونقل المعرفة والتدريب (عند الحاجة)\n"
            )

        if key == "timeline":
            base = (
                "### خطة العمل والجدول الزمني\n"
                "سنقدّم خطة زمنية تفصيلية تربط المهام بالمخرجات ومعالم المشروع (Milestones)، "
                "مع تحديد نقاط المراجعة والاعتماد لضمان الالتزام بالمدة.\n"
                "- تحديد مراحل التنفيذ ومعالم التسليم\n"
                "- تقارير تقدم دورية وربطها بالإنجاز الفعلي\n"
            )
            if user_time:
                base += f"\n**مدخل المستخدم (الجدول الزمني):**\n{user_time}\n"
            return base

        if key == "deliverables":
            return (
                "### المخرجات والتسليمات\n"
                "سنلتزم بتسليم المخرجات وفق ما ورد في الكراسة، مع توثيق كل مخرج ومعايير قبوله، وتشمل بشكل عام:\n"
                "- خطة تنفيذ تفصيلية وتقارير تقدم\n"
                "- مخرجات فنية/تشغيلية حسب نطاق المشروع\n"
                "- توثيق فني ودليل تشغيل/استخدام\n"
                "- جلسات نقل معرفة/تدريب عند الطلب\n"
            )

        if key == "compliance":
            return (
                "### الالتزام والمتطلبات النظامية\n"
                "نؤكد التزامنا بالأنظمة واللوائح ذات العلاقة وبنود الكراسة، مع تطبيق حوكمة واضحة لإدارة التغيير "
                "وضمان سرية المعلومات وجودة المخرجات ضمن الإطار النظامي.\n"
            )

        return "سيتم إعداد هذا القسم وفق متطلبات الكراسة."

    # ---------- Build draft ----------
    md = []
    md.append("# العرض الفني\n")
    md.append(f"**اسم المشروع:** {project_title}\n")
    md.append(f"**رقم المشروع (pair_id):** {pair_id}\n")

    for sec in structure:
        if not sec.get("required", False):
            continue

        key = sec["key"]
        title = sec["title"]

        # Retrieve evidence from RFP
        q_rfp = _intent_query(key, "rfp")
        rfp_hits = _retrieve("rfp", q_rfp, limit=top_k)
        rfp_ctx = _build_context(rfp_hits, max_chars=max_chars_rfp)

        # Retrieve optional support from TP
        tp_ctx = ""
        if include_tp_support:
            q_tp = _intent_query(key, "tp")
            tp_hits = _retrieve("tp", q_tp, limit=max(3, top_k // 2))
            tp_ctx = _build_context(tp_hits, max_chars=max_chars_tp)

        # Write business-style section
        section_text = _write_section(key, rfp_ctx, tp_ctx)

        md.append(f"\n## {title}\n")
        md.append(section_text.strip())

        # Attach evidence (optional)
        if include_evidence:
            if rfp_ctx:
                md.append("\nمستندات مرجعية من الكراسة (RFP)\n")
                md.append(rfp_ctx)
            if include_tp_support and tp_ctx:
                md.append("\n**مرجع أسلوبي (TP):**\n")
                md.append(tp_ctx)

    return "\n".join(md).strip()


Test Example

In [8]:
draft = generate_tech_offer_draft_with_intents(
    pair_id=1,
    project_title="مشروع تقديم خدمات استشارية تسويقية للبرامج الترويجية",
    structure=TECH_OFFER_STRUCTURE,
    client=client,
    collection_name=COLLECTION,
    embed_fn=embed,
    include_tp_support=True,
    top_k=8,
    min_score=0.30,
    scope_text="تقديم خطة تسويق شاملة للبرامج الترويجية، تشمل تحليل السوق، تحديد الجمهور المستهدف، خطة المحتوى، وخطة الحملات.",
    timeline_text="مدة التنفيذ 12 شهر تبدأ من تاريخ إشعار المباشرة، مع تسليم خطة أولية خلال 4 أسابيع وتقارير شهرية."
)

print(draft[:2500])


# العرض الفني

**اسم المشروع:** مشروع تقديم خدمات استشارية تسويقية للبرامج الترويجية

**رقم المشروع (pair_id):** 1


## 1) مقدمة وفهم عام للمشروع

مده تنفيذ المشروع 12 اشهر ميالديه تبدا من تاريخ االشعار بالمباشره، يلتزم خاللها المتعاقد باالتي: تقديم برنامجا زمنيا عام مع العرض الفني يتضمن ترتيب سير العمل والطريقه التي يقترحها لتنفيذ نطاق العمل، وكذلك علي المتعاقد ان يقدم الي الجهه او ممثل الجهه عندما يطلب منه اي معلومات تفصيليه تتعلق بالترتيبات الالزمه النجاز الخدمات التي يرغب المتعاقد في تقديمها او استعمالها او انشايها حسب االحوال. تقديم مخطط زمني الطالق البرامج التسويقيه المقترحه سواء كانت مباشره او غير مباشره، وفقا لمراحل االستهداف التي سبق ذكره ضمن نطاق العمل يكون البرنامج الزمني المفصل علي شكل رسوم بيانيه وجداول زمنيه، وسوف يكون كامال من كافه النواحي ليغطي كل النشاطات الخاصه بالعمليات واالدوات.

تقديم هيكل فريق العمل المنوط به تقديم الخدمات طوال المشروع مع تعيين مدير للمشروع وامكانيه تعيين اصحاب الخبرات حسب المواضيع المتخصصه. يلتزم المتعاقد بالمدد السابق ذكرها لتقديم الجدول الزمني 

In [9]:
draft = generate_business_style_tech_offer(
    pair_id=1,
    project_title="استشارات تسويقية",
    structure=TECH_OFFER_STRUCTURE,
    client=client,
    collection_name=COLLECTION,
    embed_fn=embed,
    include_tp_support=True,
    include_evidence=True,
    top_k=8,
    min_score=0.30
)

print(draft[:1200])


# العرض الفني

**اسم المشروع:** استشارات تسويقية

**رقم المشروع (pair_id):** 1


## 1) مقدمة وفهم عام للمشروع

نقدّم عرضنا الفني لتنفيذ المشروع وفق أفضل الممارسات، مع التركيز على تحقيق أهداف الجهة ورفع كفاءة التنفيذ وضمان جودة التسليم.
يعتمد نهجنا على فهم واضح لمتطلبات الكراسة وتحويلها إلى خطة تنفيذ قابلة للقياس، مع حوكمة واضحة وإدارة مخاطر فعّالة.

مستندات مرجعية من الكراسة (RFP)

- مده تنفيذ المشروع 12 اشهر ميلادية تبدا من تاريخ الشعار بالمباشره، يلتزم خاللها المتعاقد بالتي:
تقديم برنامجا زمنيا عام مع العرض الفني يتضمن ترتيب سير العمل والطريقه التي يقترحها لتنفيذ نطاق العمل، وكذلك
علي المتعاقد ان يقدم الي الجهه او ممثل الجهه عندما يطلب منه اي معلومات تفصيليه تتعلق بالترتيبات الالزمه النجاز
الخدمات التي يرغب المتعاقد في تقديمها او استعمالها او انشايها حسب الحوال.
- تقديم مخطط زمني الطالق البرامج التسويقيه المقترحه سواء كانت مباشره او غير مباشره، وفقا لمراحل الستهداف
التي سبق ذكره ضمن نطاق العمل
يكون البرنامج الزمني المفصل علي شكل رسوم بيانيه وجداول زمنيه، وسوف يكون كامال من كافه النوا

---------------hhhhh

In [10]:
from typing import Optional

def generate_technical_offer_one_function(
    llm_client,
    model: str,
    project_title: str,
    duration: str,
    scope: str,
    special_terms: str = "",
    temperature: float = 0.3,
    max_tokens: int = 1500,
) -> str:
    """
    Generates a structured technical offer draft (Markdown) for Saudi government tenders.
    Inputs: project title, duration, scope, and special terms.
    Output: Markdown text with fixed sections.
    """

    # -----------------------------
    # 1) SYSTEM PROMPT (fixed)
    # -----------------------------
    system_prompt = """
أنت كاتب عروض فنية محترف للمنافسات الحكومية في السعودية.
مهمتك: صياغة مسودة عرض فني “جزئي” منظم وفق الهيكل المحدد أدناه، بالاعتماد على:
1) نص الكراسة (RFP) والمقتطفات المستخرجة
2) مدخلات المستخدم (Scope / Timeline إن وجدت)
3) مرجع أسلوبي من عروض سابقة (TP) للاسترشاد بالصياغة فقط دون نسخ.

قواعد صارمة:
- لا تنسخ نصوص الكراسة حرفيًا: أعد الصياغة مع الحفاظ على المعنى.
- لا تخترع متطلبات أو أرقام أو مدد غير مذكورة.
- إذا نقصت معلومة: اكتبها كـ “يتطلب تأكيد من الجهة” بدل الاختلاق.
- التزم بلغة عربية رسمية واضحة، جُمل قصيرة، ونقاط عند الحاجة.
- تجنب الحشو والكلام العام. كل فقرة يجب أن تعكس متطلبًا أو خطة تنفيذ.
- لا تذكر أنك نموذج ذكاء اصطناعي.

الأسلوب:
- نبرة مهنية رسمية.
- استخدم عناوين واضحة وترقيم مطابق للهيكل.
- طول كل قسم 6–12 سطر كحد أقصى (إلا إذا طُلب غير ذلك).

الهيكل النهائي المطلوب إخراجه (Output):
1) مقدمة وفهم عام للمشروع
2) فهم نطاق العمل والمتطلبات
3) منهجية التنفيذ وآلية العمل
4) خطة العمل والجدول الزمني
5) المخرجات والتسليمات
6) الالتزام والمتطلبات النظامية

ملاحظة: تجاهل أي قسم غير موجود في الهيكل.
أخرج النص بصيغة Markdown بعناوين ## لكل قسم.
""".strip()

    # -----------------------------
    # 2) USER PROMPT (dynamic)
    # -----------------------------
    special_terms_block = special_terms.strip() if special_terms.strip() else "لا يوجد شروط خاصة إضافية."
    user_prompt = f"""
### بيانات المشروع
- اسم المشروع: {project_title}
- مدة التنفيذ: {duration}

### نطاق العمل (من المستخدم)
{scope}

### الشروط الخاصة
{special_terms_block}
""".strip()

    # -----------------------------
    # 3) CALL LLM
    # -----------------------------
    resp = llm_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return resp.choices[0].message.content


Test Example 